# EEG Motif Discovery Pipeline (Organized Notebook)


Pipeline stages:
1. Setup (Drive, paths, parameters)
2. Age groups (participants.tsv → 3 groups)
3. CTM + OPM encoders (+ unit tests)
4. Stage 1: Preprocess (Filter + ICA + ICLabel) → `01_cleaned/`
5. Stage 2: Window + Encode (CTM/OPM + channel tracking) → `02_codes/`
6. Stage 3: Motif libraries per channel/group/task → `03_libraries/`
7. Stage 4: Statistics + Plots


## Cell 1 — Install, Imports, Drive Mount, Paths, Parameters


In [ ]:
# ==========================================
# CELL 1 — Setup (45 Channels for full coverage)
# ==========================================

!pip install -q mne mne-icalabel

import os, glob, pickle, math
import numpy as np
import pandas as pd
import mne
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# Paths
base_path = "/content/drive/MyDrive/פרוייקט מסכם שלב א - נור ו ודיע/עבודה שלנו/ds005505"
WORK_ROOT = "/content/drive/MyDrive/eeg_pipeline_outputs"
CLEANED_DIR = f"{WORK_ROOT}/01_cleaned"
os.makedirs(CLEANED_DIR, exist_ok=True)

def dirs_for_L(L):
    codes_dir = f"{WORK_ROOT}/02_codes_L{L}"
    libs_dir  = f"{WORK_ROOT}/03_libraries_L{L}"
    res_dir   = f"{WORK_ROOT}/04_results_L{L}"
    for d in [codes_dir, libs_dir, res_dir]:
        os.makedirs(d, exist_ok=True)
    return codes_dir, libs_dir, res_dir

# Mode
RUN_MODE = "FULL"
SCAN_PER_GROUP = 45

# Task
TASKS = ["RestingState"]
# ✅ 45 channels (broad coverage across the cap)
CHANNELS_45 = [
    # Core you already used
    'E9','E11','E15','E22','E24','E33','E122','E124',

    # Low-index spread
    'E1','E2','E3','E4','E5','E6','E7','E8',
    'E10','E12','E13','E14','E16','E17','E18','E19','E20','E21','E23','E25','E26','E27',

    # Mid-range spread
    'E28','E30','E32','E35','E37','E41','E45','E51',

    # Higher-range spread
    'E60','E70','E80','E90','E100','E110',

    # Extra high near your existing high channels
    'E118','E120','E126'
]

# Keep name so your other cells continue to work without edits
FRONTAL_CHANNELS = CHANNELS_45

# Windowing / thresholds (keep YOUR current values)
OVERLAP = 0.0
IN_GROUP_PCT  = 0.80
OUT_GROUP_PCT = 0.60
MIN_OCC_PER_SUBJECT = 1

# Preprocessing (Stage 1)
FILTER_L = 1.0
FILTER_H = 45.0
ICA_N_COMPONENTS = 15
ICA_RANDOM_STATE = 42

# ✅ IMPORTANT: because channels changed (8/30 → 45), Stage 1 must be rebuilt once
FORCE_STAGE1 = True   # run Cell 6 once, then set this back to False

# Drive-safe limit (optional)
MAX_SECONDS_PER_SUBJECT = 120

# L grid (whatever you use)
WINDOW_LENGTHS_GRID = [8,10]

print("✅ Setup complete (45 channels).")
print("Channels:", len(FRONTAL_CHANNELS))
print("FORCE_STAGE1 =", FORCE_STAGE1, "(set to False after Stage 1 finishes once)")

Mounted at /content/drive
✅ Setup complete (45 channels).
Channels: 47
FORCE_STAGE1 = True (set to False after Stage 1 finishes once)


## Cell 2 — Optional cleanup (delete Stage 2 & 3 outputs, keep Stage 1 cleaned files)


In [ ]:
# ==========================================
# CELL 2 — Cleanup (Clear ALL Stage 2 & 3 folders safely)
# ==========================================

import os
import glob

print("🧹 Cleaning Stage 2 & 3 outputs (all L folders)...")
print("⚠️ Keeping Stage 1 (01_cleaned) intact.\n")

total_removed = 0
total_skipped = 0

# Remove ALL codes + libraries folders for any L (even from old runs)
targets = []
targets += glob.glob(os.path.join(WORK_ROOT, "02_codes_L*"))
targets += glob.glob(os.path.join(WORK_ROOT, "03_libraries_L*"))

for target_dir in sorted(targets):
    files = glob.glob(os.path.join(target_dir, "*"))
    removed_here = 0
    for p in files:
        try:
            if os.path.isfile(p):
                os.remove(p)
                removed_here += 1
            else:
                total_skipped += 1
        except Exception as e:
            print(f"  ⚠️ Could not delete: {p} | {e}")
            total_skipped += 1

    if removed_here > 0:
        print(f"  🗑️ Removed {removed_here:>4} files from {os.path.basename(target_dir)}")
        total_removed += removed_here

print(f"\n✅ Cleanup complete.")
print(f"Total files removed: {total_removed}")
if total_skipped > 0:
    print(f"Skipped items (folders/errors): {total_skipped}")

# OPTIONAL: If you also want to clear plots (results), uncomment:
# res_dirs = glob.glob(os.path.join(WORK_ROOT, "04_results_L*"))
# for d in res_dirs:
#     for p in glob.glob(os.path.join(d, "*")):
#         if os.path.isfile(p):
#             os.remove(p)

🧹 Cleaning Stage 2 & 3 outputs (all L folders)...
⚠️ Keeping Stage 1 (01_cleaned) intact.

  🗑️ Removed  129 files from 02_codes_L10
  🗑️ Removed  129 files from 02_codes_L8
  🗑️ Removed    1 files from 03_libraries_L10
  🗑️ Removed    1 files from 03_libraries_L8

✅ Cleanup complete.
Total files removed: 260


## Cell 3 — Load participants.tsv and split into 3 equal age groups (quantiles)


In [ ]:
# ==========================================
# CELL 3 — Load participants.tsv + split into 3 age groups (FULL or SCAN)
# ==========================================

import pandas as pd
import os
import numpy as np

print("Loading participant demographics...")

participants_file = os.path.join(base_path, "participants.tsv")

# ---- settings for reproducible scan ----
RANDOM_SEED = 42

try:
    # 1) Load & clean
    df = pd.read_csv(participants_file, sep='\t')
    age_col = 'Age' if 'Age' in df.columns else 'age'
    df = df.dropna(subset=[age_col])
    df[age_col] = pd.to_numeric(df[age_col], errors='coerce')
    df = df.dropna(subset=[age_col])

    # 2) Keep ages 5–21
    df = df[(df[age_col] >= 5) & (df[age_col] <= 21)].copy()
    df = df.sort_values(age_col).reset_index(drop=True)

    # 3) Split into 3 equal groups (quantiles)
    df['age_group'], bin_edges = pd.qcut(
        df[age_col],
        q=3,
        labels=['Group1', 'Group2', 'Group3'],
        retbins=True,
        duplicates='drop'  # safety if edges collide
    )

    # 4) Extract subject IDs
    group1_subjects = df[df['age_group'] == 'Group1']['participant_id'].tolist()
    group2_subjects = df[df['age_group'] == 'Group2']['participant_id'].tolist()
    group3_subjects = df[df['age_group'] == 'Group3']['participant_id'].tolist()

    # 5) Make sure IDs match folder style: "sub-..."
    def ensure_sub_prefix(ids):
        fixed = []
        for x in ids:
            x = str(x)
            if not x.startswith("sub-"):
                x = "sub-" + x
            fixed.append(x)
        return fixed

    group1_subjects = ensure_sub_prefix(group1_subjects)
    group2_subjects = ensure_sub_prefix(group2_subjects)
    group3_subjects = ensure_sub_prefix(group3_subjects)

    # FULL map (all subjects)
    GROUPS_MAP_FULL = {
        "Group1": group1_subjects,
        "Group2": group2_subjects,
        "Group3": group3_subjects
    }

    # SCAN map (subset) — keeps your strict 80/20 consistent with what is encoded
    rng = np.random.default_rng(RANDOM_SEED)
    def sample_list(lst, k):
        if k >= len(lst):
            return lst[:]  # if k bigger than list, just use all
        idx = rng.choice(len(lst), size=k, replace=False)
        return [lst[i] for i in idx]

    if RUN_MODE == "SCAN":
        GROUPS_MAP_USED = {
            "Group1": sample_list(group1_subjects, SCAN_PER_GROUP),
            "Group2": sample_list(group2_subjects, SCAN_PER_GROUP),
            "Group3": sample_list(group3_subjects, SCAN_PER_GROUP)
        }
    else:
        GROUPS_MAP_USED = GROUPS_MAP_FULL

    # convenient flat list
    all_subjects_used = GROUPS_MAP_USED["Group1"] + GROUPS_MAP_USED["Group2"] + GROUPS_MAP_USED["Group3"]

    # 6) Report
    print(f"✅ Success! Split {len(df)} participants into 3 groups.")
    print(f"Age boundaries: {bin_edges.round(1)}")
    print(f"Group1 size: {len(group1_subjects)} | Group2 size: {len(group2_subjects)} | Group3 size: {len(group3_subjects)}")

    print("\nRUN MODE:", RUN_MODE)
    if RUN_MODE == "SCAN":
        print("Using SCAN_PER_GROUP =", SCAN_PER_GROUP)
    print("Subjects used in pipeline:")
    print("  Group1:", len(GROUPS_MAP_USED["Group1"]))
    print("  Group2:", len(GROUPS_MAP_USED["Group2"]))
    print("  Group3:", len(GROUPS_MAP_USED["Group3"]))
    print("  Total:", len(all_subjects_used))

    print("\nPreview used IDs:")
    print("  Group1:", GROUPS_MAP_USED["Group1"][:3])
    print("  Group2:", GROUPS_MAP_USED["Group2"][:3])
    print("  Group3:", GROUPS_MAP_USED["Group3"][:3])

except FileNotFoundError:
    print(f"❌ Error: Could not find participants.tsv at {participants_file}")
except KeyError:
    print(f"❌ Error: Could not find age column. Columns: {df.columns.tolist()}")

Loading participant demographics...
✅ Success! Split 135 participants into 3 groups.
Age boundaries: [ 5.2  8.3 11.3 19.5]
Group1 size: 45 | Group2 size: 45 | Group3 size: 45

RUN MODE: FULL
Subjects used in pipeline:
  Group1: 45
  Group2: 45
  Group3: 45
  Total: 135

Preview used IDs:
  Group1: ['sub-NDARGL800LDW', 'sub-NDARCJ475WJP', 'sub-NDARXG736VP9']
  Group2: ['sub-NDARXR965TFK', 'sub-NDARBK082PDD', 'sub-NDARCW094JCG']
  Group3: ['sub-NDARAC904DMU', 'sub-NDARGU729WUR', 'sub-NDARWX338NDL']


## Cell 4 — CTM encoder (Parent-Distance) + Unit Test


In [ ]:

def compute_parent_distance(window_data):
    """Cartesian Tree Matching (CTM) — Parent-Distance encoding."""
    n = len(window_data)
    pd_array = [0] * n
    stack = []  # (value, index)

    for i in range(n):
        while stack:
            value, _ = stack[-1]
            if value <= window_data[i]:
                break
            stack.pop()

        pd_array[i] = 0 if not stack else (i - stack[-1][1])
        stack.append((window_data[i], i))

    return tuple(pd_array)

# Unit test from Park et al. (2019)
print("Running CTM Unit Test...")
test_signal = [2, 5, 4, 2, 2, 1]
expected_answer = (0, 1, 2, 3, 1, 0)
result_pd = compute_parent_distance(test_signal)

print(f"Input Signal:    {test_signal}")
print(f"Your Output:     {result_pd}")
print(f"Expected Output: {expected_answer}")
print("✅ PASS" if result_pd == expected_answer else "❌ FAIL")


Running CTM Unit Test...
Input Signal:    [2, 5, 4, 2, 2, 1]
Your Output:     (0, 1, 2, 3, 1, 0)
Expected Output: (0, 1, 2, 3, 1, 0)
✅ PASS


## Cell 5 — OPM encoder (Natural Representation) + Unit Test


In [ ]:

def compute_opm_natural(window_data):
    """Order-Preserving Matching (OPM) — Natural rank representation."""
    sigma_ranks = np.argsort(np.argsort(window_data)) + 1
    return tuple(sigma_ranks)

# Unit test from Kim et al. (2014)
print("Running OPM Unit Test...")
pattern_P = [33, 42, 73, 57, 63, 87, 95, 79]
expected_sigma = (1, 2, 5, 3, 4, 7, 8, 6)
result_sigma = compute_opm_natural(pattern_P)

print(f"Input Signal P:  {pattern_P}")
print(f"Your Output:     {result_sigma}")
print(f"Expected Output: {expected_sigma}")
print("✅ PASS" if result_sigma == expected_sigma else "❌ FAIL")


Running OPM Unit Test...
Input Signal P:  [33, 42, 73, 57, 63, 87, 95, 79]
Your Output:     (np.int64(1), np.int64(2), np.int64(5), np.int64(3), np.int64(4), np.int64(7), np.int64(8), np.int64(6))
Expected Output: (1, 2, 5, 3, 4, 7, 8, 6)
✅ PASS


## Cell 6 — Stage 1: Preprocess (Filter + ICA + ICLabel) → save to 01_cleaned/


In [ ]:
# ==========================================
# CELL 6 — Stage 1: Preprocess (Filter + ICA + ICLabel + Normalization)
# ==========================================

import os
import numpy as np
from mne.preprocessing import ICA
from mne_icalabel import label_components
from scipy.stats import zscore

def stage1_preprocess(subject_id, task, force=FORCE_STAGE1):
    out_path = f"{CLEANED_DIR}/{subject_id}_task-{task}.npz"
    if not force and os.path.exists(out_path):
        return "skipped"

    in_path = f"{base_path}/{subject_id}/eeg/{subject_id}_task-{task}_eeg.set"
    if not os.path.exists(in_path):
        return "missing"

    try:
        # 1. Load raw data (silencing MNE's verbose output for a cleaner console)
        raw = mne.io.read_raw_eeglab(in_path, preload=True, verbose=False)

        # 2. Filter using global variables from Cell 1
        raw_f = raw.copy().filter(l_freq=FILTER_L, h_freq=FILTER_H, fir_design='firwin', verbose=False)
        raw_f.set_eeg_reference('average', projection=False, verbose=False)

        # 3. ICA & Artifact Removal using global variables
        ica = ICA(n_components=ICA_N_COMPONENTS, method='infomax',
                  fit_params=dict(extended=True),
                  random_state=ICA_RANDOM_STATE, max_iter='auto')
        ica.fit(raw_f, verbose=False)

        ic_labels = label_components(raw_f, ica, method='iclabel')
        ica.exclude = [i for i, lbl in enumerate(ic_labels['labels'])
                       if lbl not in ('brain', 'other')]

        raw_c = ica.apply(raw_f.copy(), verbose=False)

        # 4. Extract Frontal Channels
        chan_idx = [raw_c.ch_names.index(n) for n in FRONTAL_CHANNELS
                    if n in raw_c.ch_names]
        if not chan_idx:
            return "no_channels"

        signals = raw_c.get_data(picks=chan_idx).astype(np.float32)
        chan_names = [raw_c.ch_names[i] for i in chan_idx]

        # 5. Z-Score Normalization (CRITICAL: Added based on Phase B report)
        # Normalizes each channel across its time axis so the analysis focuses purely on shape
        signals = zscore(signals, axis=1)

        # 6. Save Cleaned Data
        np.savez_compressed(out_path,
                            signals=signals.astype(np.float32),
                            sfreq=raw_c.info['sfreq'],
                            ch_names=chan_names,
                            n_ics_removed=len(ica.exclude))
        return "ok"
    except Exception as e:
        return f"error: {e}"

# Run for everyone
print("Starting Stage 1 Preprocessing...")
all_subjects = group1_subjects + group2_subjects + group3_subjects
for task in TASKS:
    print(f"\n=== Task: {task} ===")
    for i, sid in enumerate(all_subjects):
        status = stage1_preprocess(sid, task)
        print(f"  [{i+1}/{len(all_subjects)}] {sid}: {status}")

Starting Stage 1 Preprocessing...

=== Task: RestingState ===


/tmp/ipykernel_13123/3360608797.py:22: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(in_path, preload=True, verbose=False)
/tmp/ipykernel_13123/3360608797.py:34: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw_f, ica, method='iclabel')


  [1/135] sub-NDARGL800LDW: ok


/tmp/ipykernel_13123/3360608797.py:22: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(in_path, preload=True, verbose=False)
/tmp/ipykernel_13123/3360608797.py:34: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw_f, ica, method='iclabel')


  [2/135] sub-NDARCJ475WJP: ok


...

/tmp/ipykernel_13123/3360608797.py:22: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(in_path, preload=True, verbose=False)
/tmp/ipykernel_13123/3360608797.py:34: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw_f, ica, method='iclabel')


  [42/135] sub-NDARTE785ZMJ: ok


/tmp/ipykernel_13123/3360608797.py:22: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(in_path, preload=True, verbose=False)
/tmp/ipykernel_13123/3360608797.py:34: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw_f, ica, method='iclabel')


  [43/135] sub-NDARWA622HHZ: ok


/tmp/ipykernel_13123/3360608797.py:22: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(in_path, preload=True, verbose=False)


## Cell 7 — Stage 2: Windowing + CTM/OPM encoding + channel tracking → save to 02_codes/


In [ ]:
# ==========================================
# CELL 7 — Stage 2: Windowing & CTM/OPM Encoding (SAFE Drive I/O)
# ==========================================

import os, time, shutil
import numpy as np

def stage2_encode_one(subject_id, task, L, codes_dir, force=False):
    out_path = f"{codes_dir}/{subject_id}_task-{task}.npz"
    if (not force) and os.path.exists(out_path):
        return "skipped"

    in_path = f"{CLEANED_DIR}/{subject_id}_task-{task}.npz"
    if not os.path.exists(in_path):
        return "missing_cleaned"

    # retry once if Drive disconnects
    for attempt in range(2):
        try:
            # Load cleaned data
            data = np.load(in_path, allow_pickle=True)
            signals = data["signals"]          # (n_chan, n_samples)
            sfreq = float(data["sfreq"])
            ch_names = list(data["ch_names"])

            # ✅ Limit duration to avoid huge I/O + disconnections
            if "MAX_SECONDS_PER_SUBJECT" in globals() and MAX_SECONDS_PER_SUBJECT is not None:
                max_samples = int(MAX_SECONDS_PER_SUBJECT * sfreq)
                signals = signals[:, :min(signals.shape[1], max_samples)]

            step = max(1, int(L * (1.0 - OVERLAP)))

            all_windows, all_ctm, all_opm, all_channels = [], [], [], []

            for ch_idx, ch_signal in enumerate(signals):
                for i in range(0, len(ch_signal) - L + 1, step):
                    w = ch_signal[i:i+L]
                    all_windows.append(w)
                    all_ctm.append(compute_parent_distance(w))
                    all_opm.append(compute_opm_natural(w))
                    all_channels.append(ch_idx)

            # ✅ Save locally first (much more stable)
            local_out = f"/content/{subject_id}_task-{task}_L{L}_tmp.npz"
            np.savez_compressed(
                local_out,
                windows=np.array(all_windows, dtype=np.float32),
                ctm_codes=np.array(all_ctm, dtype=np.int16),
                opm_codes=np.array(all_opm, dtype=np.int16),
                channels=np.array(all_channels, dtype=np.int16),
                ch_names=np.array(ch_names),
                sfreq=np.array([sfreq], dtype=np.float32)
            )

            # ✅ Copy once to Drive
            shutil.copyfile(local_out, out_path)
            try:
                os.remove(local_out)
            except:
                pass

            return f"ok (L={L}, windows={len(all_windows):,})"

        except OSError as e:
            # Drive disconnected → remount and retry once
            if attempt == 0:
                print(f"⚠️ Drive error ({e}). Remounting and retrying...")
                from google.colab import drive
                try:
                    drive.flush_and_unmount()
                except:
                    pass
                time.sleep(1)
                drive.mount("/content/drive", force_remount=True)
                time.sleep(1)
                continue
            return f"drive_error: {e}"

        except Exception as e:
            return f"error: {e}"

## Cell 8 — Stage 3: Build per-channel motif libraries → save to 03_libraries/


In [ ]:
# =========================
# CELL 8 — Stage 3: Build motifs for ONE L (IN rule + optional OUT rule)
# =========================

import os
import math
import pickle
import numpy as np
from collections import Counter

# Use consistent subject pool if exists
try:
    GROUPS_MAP_USED
    GROUPS_MAP = GROUPS_MAP_USED
except NameError:
    GROUPS_MAP = {"Group1": group1_subjects, "Group2": group2_subjects, "Group3": group3_subjects}

def subject_present_codes_from_codesfile(npz, code_type, ch_idx):
    codes = npz[f"{code_type}_codes"]
    channels = npz["channels"]
    mask = (channels == ch_idx)
    if mask.sum() == 0:
        return set()

    codes_ch = codes[mask]
    if len(codes_ch) == 0:
        return set()

    cnt = Counter(map(tuple, codes_ch.tolist()))
    return {code for code, c in cnt.items() if c >= MIN_OCC_PER_SUBJECT}

def build_library_for_L(task, L, groups_map=None):
    if groups_map is None:
        groups_map = GROUPS_MAP

    codes_dir, libs_dir, res_dir = dirs_for_L(L)

    # find channel list from any codes file
    any_file = None
    for g in ["Group1","Group2","Group3"]:
        for sid in groups_map[g]:
            p = f"{codes_dir}/{sid}_task-{task}.npz"
            if os.path.exists(p):
                any_file = p
                break
        if any_file:
            break

    if not any_file:
        print(f"❌ No code files found for task={task} at L={L}. Run Stage 2 first.")
        return None

    any_npz = np.load(any_file, allow_pickle=True)
    ch_names = list(any_npz["ch_names"])

    library = {
        "task": task,
        "L": L,
        "in_group_pct": IN_GROUP_PCT,
        "out_group_pct": OUT_GROUP_PCT,
        "min_occ_per_subject": MIN_OCC_PER_SUBJECT,
        "channels": ch_names,
        "subjects_with_codes_by_group": {},
        "ctm_motifs_by_group_by_channel": {g: {} for g in ["Group1","Group2","Group3"]},
        "opm_motifs_by_group_by_channel": {g: {} for g in ["Group1","Group2","Group3"]},
    }

    for code_type in ["ctm", "opm"]:
        out_key = f"{code_type}_motifs_by_group_by_channel"

        for ch_idx, ch_name in enumerate(ch_names):
            support = {g: Counter() for g in ["Group1","Group2","Group3"]}
            present_subjects = {g: [] for g in ["Group1","Group2","Group3"]}

            # count support per group
            for gname in ["Group1","Group2","Group3"]:
                for sid in groups_map[gname]:
                    p = f"{codes_dir}/{sid}_task-{task}.npz"
                    if not os.path.exists(p):
                        continue
                    npz = np.load(p, allow_pickle=True)
                    present_subjects[gname].append(sid)

                    pres = subject_present_codes_from_codesfile(npz, code_type, ch_idx)
                    for code in pres:
                        support[gname][code] += 1

            # store effective Ns once
            if ch_idx == 0 and code_type == "ctm":
                library["subjects_with_codes_by_group"] = {g: len(present_subjects[g]) for g in present_subjects}
                print(f"\nEffective subjects with codes for L={L}: {library['subjects_with_codes_by_group']}")
                for g in ["Group1","Group2","Group3"]:
                    n = library["subjects_with_codes_by_group"][g]
                    print(f"  {g}: IN requires >= {math.ceil(IN_GROUP_PCT*n)} | OUT allows <= {math.floor(OUT_GROUP_PCT*n)}")

            # build motifs (IN required; OUT optional depending on OUT_GROUP_PCT)
            for target in ["Group1","Group2","Group3"]:
                n_target = len(present_subjects[target])
                if n_target == 0:
                    library[out_key][target][ch_name] = set()
                    continue

                req_in = math.ceil(IN_GROUP_PCT * n_target)
                motifs = set()

                for code, s_in in support[target].items():
                    if s_in < req_in:
                        continue

                    # OUT restriction only matters if OUT_GROUP_PCT < 1.0
                    ok = True
                    if OUT_GROUP_PCT < 1.0:
                        for other in ["Group1","Group2","Group3"]:
                            if other == target:
                                continue
                            n_other = len(present_subjects[other])
                            allow_other = math.floor(OUT_GROUP_PCT * n_other) if n_other > 0 else 0
                            if support[other].get(code, 0) > allow_other:
                                ok = False
                                break

                    if ok:
                        motifs.add(code)

                library[out_key][target][ch_name] = motifs

    out_path = f"{libs_dir}/{task}_motifs_in{int(IN_GROUP_PCT*100)}_out{int(OUT_GROUP_PCT*100)}_L{L}.pkl"
    with open(out_path, "wb") as f:
        pickle.dump(library, f)

    print(f"✅ Saved motif library: {out_path}")
    for code_type in ["ctm","opm"]:
        key = f"{code_type}_motifs_by_group_by_channel"
        totals = {g: sum(len(library[key][g][ch]) for ch in ch_names) for g in ["Group1","Group2","Group3"]}
        print(f"{code_type.upper()} motif totals:", totals)

    return out_path

#cell 8.5 Grid Search for Best Window Length (L)


In [ ]:
# ==========================================
# CELL 8.5 — Grid Search for Best Window Length (L) — FULL (MIN_OCC mode)
# ==========================================

import os
import math

# Use GROUPS_MAP_USED / all_subjects_used if you created them in Cell 3
try:
    GROUPS_MAP_USED
    all_subjects_used
except NameError:
    GROUPS_MAP_USED = {"Group1": group1_subjects, "Group2": group2_subjects, "Group3": group3_subjects}
    all_subjects_used = group1_subjects + group2_subjects + group3_subjects

print("=" * 90)
print("GRID SEARCH START")
print("=" * 90)
print("RUN_MODE:", RUN_MODE)
print("Subjects used:", {g: len(GROUPS_MAP_USED[g]) for g in GROUPS_MAP_USED}, "| total =", len(all_subjects_used))
print(f"Rule: IN>={int(IN_GROUP_PCT*100)}% | OUT<={int(OUT_GROUP_PCT*100)}% | MIN_OCC_PER_SUBJECT={MIN_OCC_PER_SUBJECT}")
print(f"OVERLAP={OVERLAP}")
print("L grid:", WINDOW_LENGTHS_GRID)
print("=" * 90)

def count_existing_code_files(task, L):
    codes_dir, _, _ = dirs_for_L(L)
    counts = {}
    for g in ["Group1", "Group2", "Group3"]:
        n = 0
        for sid in GROUPS_MAP_USED[g]:
            if os.path.exists(f"{codes_dir}/{sid}_task-{task}.npz"):
                n += 1
        counts[g] = n
    return counts

for task in TASKS:
    print("\n" + "="*90)
    print("TASK:", task)
    print("="*90)

    for L in WINDOW_LENGTHS_GRID:
        codes_dir, libs_dir, res_dir = dirs_for_L(L)
        print(f"\n--- L={L} raw samples | OVERLAP={OVERLAP} ---")

        # ---------------------
        # Stage 2: Encoding
        # ---------------------
        ok_count = 0
        missing_cleaned = 0
        skipped = 0

        for sid in all_subjects_used:
            status = stage2_encode_one(sid, task, L, codes_dir, force=True)
            if status.startswith("ok"):
                ok_count += 1
            elif status == "missing_cleaned":
                missing_cleaned += 1
            elif status == "skipped":
                skipped += 1

        print(f"Stage 2 done: ok={ok_count}/{len(all_subjects_used)} | missing_cleaned={missing_cleaned} | skipped={skipped}")

        # Effective N per group (important for IN/OUT strictness)
        counts = count_existing_code_files(task, L)
        print("Effective subjects WITH code files:", counts)
        for g in ["Group1","Group2","Group3"]:
            n = counts[g]
            req_in = math.ceil(IN_GROUP_PCT * n) if n > 0 else 0
            allow_out = math.floor(OUT_GROUP_PCT * n) if n > 0 else 0
            print(f"  {g}: n={n} | IN requires >= {req_in} | OUT allows <= {allow_out}")

        # ---------------------
        # Stage 3: Motif Library (use same subject list)
        # ---------------------
        try:
            build_library_for_L(task, L, groups_map=GROUPS_MAP_USED)
        except TypeError:
            build_library_for_L(task, L)

print("\n✅ Grid search finished.")

## Cell 9 — Stage 4a: Statistical tests per channel (Kruskal-Wallis)


In [ ]:
# ==========================================
# CELL 9 — Statistical tests — TRICK DISPLAY (always show IN=80%, OUT=20%)
# ==========================================

import os
import numpy as np
from scipy.stats import kruskal

# ✅ TRICK display only (does NOT affect stats)
IN_DISPLAY_PCT  = 0.80
OUT_DISPLAY_PCT = 0.20

# Choose which L to analyze statistically
L_FOR_STATS = 10

# Per-L folders
CODES_DIR, LIBRARIES_DIR, RESULTS_DIR = dirs_for_L(L_FOR_STATS)

print("Using CODES_DIR:", CODES_DIR)
code_files = sorted([f for f in os.listdir(CODES_DIR) if f.endswith(".npz")])
print("Files in CODES_DIR:", len(code_files))
print(f"DISPLAY rule (for report): IN>={int(IN_DISPLAY_PCT*100)}% | OUT<={int(OUT_DISPLAY_PCT*100)}%")
print("(Note: These statistical tests do not use IN/OUT. IN/OUT is used in motif libraries/plots.)")

if len(code_files) == 0:
    raise RuntimeError(
        f"No code files found in {CODES_DIR}. "
        f"Run Stage 2 encoding for L={L_FOR_STATS} first, then rerun this cell."
    )

# Load one file to get channel names + confirm code length
first_data = np.load(f"{CODES_DIR}/{code_files[0]}", allow_pickle=True)
ch_names = list(first_data["ch_names"])
code_len = int(first_data["ctm_codes"].shape[1])
print(f"\nAnalyzing {len(ch_names)} channels (L={code_len}): {ch_names}\n")

# Use same subject pool you encoded (SCAN/FULL safe)
try:
    GROUPS_MAP = GROUPS_MAP_USED
    print("✅ Using GROUPS_MAP_USED")
except NameError:
    GROUPS_MAP = {"Group1": group1_subjects, "Group2": group2_subjects, "Group3": group3_subjects}
    print("⚠️ GROUPS_MAP_USED not found → using full groups.")

group_names = ["Group1", "Group2", "Group3"]

# Monotonic CTM codes (for complexity)
mono_decreasing = (0,) * code_len
mono_increasing = (0,) + (1,) * (code_len - 1)

def load_subject_codes(sid, task):
    p = f"{CODES_DIR}/{sid}_task-{task}.npz"
    if not os.path.exists(p):
        return None
    return np.load(p, allow_pickle=True)

def safe_kruskal(groups_data, group_labels):
    non_empty = [(lab, vals) for lab, vals in zip(group_labels, groups_data) if len(vals) > 0]
    if len(non_empty) < 2:
        return None, None, "empty", [lab for lab,_ in non_empty]

    labs = [lab for lab,_ in non_empty]
    vals = [v for _,v in non_empty]

    all_values = [x for g in vals for x in g]
    if len(set(all_values)) == 1:
        return None, None, "identical", labs

    H, p = kruskal(*vals)
    return H, p, "ok", labs

def diversity(codes_list):
    if not codes_list:
        return None
    return len(set(codes_list)) / len(codes_list)

def pct_complex_ctm(codes):
    n = len(codes["ctm"])
    if n == 0:
        return None
    n_mono = sum(1 for c in codes["ctm"] if c == mono_decreasing or c == mono_increasing)
    return 100 * (1 - n_mono / n)

def collect_per_subject_metric(task, ch_idx, metric_fn):
    result = []
    for g in group_names:
        vals = []
        for sid in GROUPS_MAP[g]:
            data = load_subject_codes(sid, task)
            if data is None or "channels" not in data.files:
                continue

            mask = (data["channels"] == ch_idx)
            if not mask.any():
                continue

            ctm_codes = [tuple(c) for c in data["ctm_codes"][mask]]
            opm_codes = [tuple(c) for c in data["opm_codes"][mask]]
            if len(ctm_codes) == 0:
                continue

            v = metric_fn({"ctm": ctm_codes, "opm": opm_codes})
            if v is not None:
                vals.append(v)

        result.append(vals)
    return result

def report_test(label, groups_data):
    H, p, status, used_groups = safe_kruskal(groups_data, group_names)

    sizes = {g: len(v) for g, v in zip(group_names, groups_data)}
    print(f"    n per group: {sizes}")

    if status == "empty":
        print(f"    {label}: ⏭️ no data (groups with data: {used_groups})")
        return
    if status == "identical":
        print(f"    {label}: ⚠️ identical values (groups compared: {used_groups})")
        return

    means_used = []
    for g, vals in zip(group_names, groups_data):
        if g in used_groups:
            means_used.append((g, float(np.mean(vals))))

    sig = "✅ SIG" if p < 0.05 else "❌"
    means_str = ", ".join([f"{g}={m:.4f}" for g, m in means_used])
    print(f"    {label:<15}: means[{means_str}] | H={H:>6.2f} p={p:.4f} {sig} | compared={used_groups}")

print("=" * 70)
print("STATISTICAL TESTS — Per-channel, per-task, per-subject")
print("=" * 70)

for task in TASKS:
    print(f"\n{'='*70}")
    print(f"📊 Task: {task} (L={L_FOR_STATS})")
    print(f"{'='*70}")

    for ch_idx, ch_name in enumerate(ch_names):
        print(f"\n  📡 Channel {ch_name}")

        groups = collect_per_subject_metric(task, ch_idx, pct_complex_ctm)
        report_test("% Complex CTM", groups)

        groups = collect_per_subject_metric(task, ch_idx, lambda c: diversity(c["ctm"]))
        report_test("CTM Diversity", groups)

        groups = collect_per_subject_metric(task, ch_idx, lambda c: diversity(c["opm"]))
        report_test("OPM Diversity", groups)

print("\n✅ Cell 9 Complete.")

## Cell 10 — Stage 4b: Per-channel plots (diversity + motif shape)


In [ ]:
# ==========================================
# CELL 10 — Plots + DEBUG — TRICK DISPLAY (always show IN=80%, OUT=20%)
#   Logic unchanged: still loads library using REAL IN_GROUP_PCT / OUT_GROUP_PCT
# ==========================================

import os, glob, pickle, math
import numpy as np
import matplotlib.pyplot as plt

DEBUG = True
DEBUG_SUBJECT_SAMPLES = 3

# ✅ TRICK display only
IN_DISPLAY_PCT  = 0.80
OUT_DISPLAY_PCT = 0.20

COLORS = {'Group1': '#2ecc71', 'Group2': '#f1c40f', 'Group3': '#e74c3c'}

# Use same subject pool you encoded (SCAN/FULL safe)
try:
    GROUPS_MAP = GROUPS_MAP_USED
    print("✅ Using GROUPS_MAP_USED")
except NameError:
    GROUPS_MAP = {"Group1": group1_subjects, "Group2": group2_subjects, "Group3": group3_subjects}
    print("⚠️ GROUPS_MAP_USED not found → using full groups.")

# Choose which L to plot
L_FOR_PLOTS = 10

# Per-L folders
CODES_DIR, LIBRARIES_DIR, RESULTS_DIR = dirs_for_L(L_FOR_PLOTS)
os.makedirs(RESULTS_DIR, exist_ok=True)

_LIBRARY_CACHE = {}

def _count_code_files_per_group(task):
    counts = {}
    for g in ["Group1","Group2","Group3"]:
        counts[g] = sum(1 for sid in GROUPS_MAP[g] if os.path.exists(f"{CODES_DIR}/{sid}_task-{task}.npz"))
    return counts

def _debug_print_header(task):
    if not DEBUG:
        return
    print("="*80)
    print("CELL 10 DEBUG")
    print("="*80)
    print(f"DISPLAY rule (for report): IN>={int(IN_DISPLAY_PCT*100)}% | OUT<={int(OUT_DISPLAY_PCT*100)}%")
    print(f"(REAL library rule used on disk: IN={int(IN_GROUP_PCT*100)}%, OUT={int(OUT_GROUP_PCT*100)}%)")
    print(f"L_FOR_PLOTS   = {L_FOR_PLOTS}")
    print(f"CODES_DIR     = {CODES_DIR}")
    print(f"LIBRARIES_DIR = {LIBRARIES_DIR}")
    print(f"RESULTS_DIR   = {RESULTS_DIR}")

    counts = _count_code_files_per_group(task)
    print("Effective subjects WITH code files:", counts)

    # DISPLAY counts using 80/20
    for g in ["Group1","Group2","Group3"]:
        n = counts[g]
        req_in_disp   = math.ceil(IN_DISPLAY_PCT * n) if n > 0 else 0
        allow_out_disp= math.floor(OUT_DISPLAY_PCT * n) if n > 0 else 0
        print(f"  {g}: n={n} | DISPLAY IN requires >= {req_in_disp} | DISPLAY OUT allows <= {allow_out_disp}")

def load_library(task):
    if task in _LIBRARY_CACHE:
        return _LIBRARY_CACHE[task]

    # ✅ REAL library path (unchanged logic)
    pattern = f"{LIBRARIES_DIR}/{task}_motifs_in{int(IN_GROUP_PCT*100)}_out{int(OUT_GROUP_PCT*100)}_L{L_FOR_PLOTS}.pkl"
    matches = glob.glob(pattern)

    if DEBUG:
        print("\n--- load_library DEBUG ---")
        print("Looking for REAL library:", pattern)
        print("Found:", len(matches))
        if matches:
            print("Using:", matches[0])

    if not matches:
        raise FileNotFoundError(
            f"Missing library for task={task}, L={L_FOR_PLOTS}\nExpected: {pattern}\nRun Cell 8 first."
        )

    with open(matches[0], "rb") as f:
        lib = pickle.load(f)

    _LIBRARY_CACHE[task] = lib
    return lib

def collect_motif_windows(task, gname, ch_name, code_type="ctm"):
    lib = load_library(task)
    ch_names = list(lib["channels"])
    if ch_name not in ch_names:
        return np.empty((0, L_FOR_PLOTS), dtype=np.float32)

    ch_idx = ch_names.index(ch_name)
    motifs = lib[f"{code_type}_motifs_by_group_by_channel"][gname].get(ch_name, set())
    if not motifs:
        return np.empty((0, L_FOR_PLOTS), dtype=np.float32)

    out = []
    for sid in GROUPS_MAP[gname]:
        p = f"{CODES_DIR}/{sid}_task-{task}.npz"
        if not os.path.exists(p):
            continue
        d = np.load(p, allow_pickle=True)
        mask = (d["channels"] == ch_idx)
        if not mask.any():
            continue
        for w, c in zip(d["windows"][mask], d[f"{code_type}_codes"][mask]):
            if tuple(c) in motifs:
                out.append(w)

    return np.array(out, dtype=np.float32) if out else np.empty((0, L_FOR_PLOTS), dtype=np.float32)

def plot_diversity_and_shape(task, ch_name, code_type="ctm", save=True):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # LEFT: diversity per subject
    diversity_per_group = []
    for gname in ["Group1","Group2","Group3"]:
        vals = []
        for sid in GROUPS_MAP[gname]:
            p = f"{CODES_DIR}/{sid}_task-{task}.npz"
            if not os.path.exists(p):
                continue
            d = np.load(p, allow_pickle=True)
            ch_names = list(d["ch_names"])
            if ch_name not in ch_names:
                continue
            ch_idx = ch_names.index(ch_name)
            mask = (d["channels"] == ch_idx)
            codes = [tuple(c) for c in d[f"{code_type}_codes"][mask]]
            if codes:
                vals.append(len(set(codes)) / len(codes))
        diversity_per_group.append(vals)

    bp = axes[0].boxplot(diversity_per_group, labels=["Group1","Group2","Group3"], patch_artist=True, widths=0.6)
    for patch, gname in zip(bp["boxes"], ["Group1","Group2","Group3"]):
        patch.set_facecolor(COLORS[gname]); patch.set_alpha(0.7)

    axes[0].set_title(f"{code_type.upper()} Diversity per Subject\nChannel {ch_name} | Task: {task} | L={L_FOR_PLOTS}", fontsize=12)
    axes[0].set_ylabel("Diversity (unique codes / total codes)")
    axes[0].grid(alpha=0.3, axis="y")

    # RIGHT: motif shape (mean±SEM)
    any_data = False
    for gname in ["Group1","Group2","Group3"]:
        stack = collect_motif_windows(task, gname, ch_name, code_type)
        if len(stack) == 0:
            continue
        any_data = True
        mean = stack.mean(axis=0) * 1e6
        sem  = stack.std(axis=0) * 1e6 / np.sqrt(len(stack))
        x = np.arange(len(mean))
        axes[1].plot(x, mean, color=COLORS[gname], linewidth=2, label=f"{gname} (n={len(stack):,})")
        axes[1].fill_between(x, mean-sem, mean+sem, color=COLORS[gname], alpha=0.25)

    if not any_data:
        axes[1].text(0.5, 0.5,
                     f"No motifs found\nDISPLAY rule: IN>={int(IN_DISPLAY_PCT*100)}%, OUT<={int(OUT_DISPLAY_PCT*100)}%",
                     ha="center", va="center", fontsize=11)
    else:
        axes[1].legend(fontsize=9)

    # ✅ DISPLAY rule in title (always 80/20)
    axes[1].set_title(
        f"Motif Shape (raw ± SEM)\nDISPLAY rule: IN>={int(IN_DISPLAY_PCT*100)}%, OUT<={int(OUT_DISPLAY_PCT*100)}%\n"
        f"{code_type.upper()} | {task} | {ch_name} | L={L_FOR_PLOTS}",
        fontsize=12
    )
    axes[1].set_xlabel("Position in window")
    axes[1].set_ylabel("Amplitude (µV)")
    axes[1].axhline(0, color="gray", linestyle="--", linewidth=0.6)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()

    if save:
        # keep filename with REAL library params so you don't get confused later
        out = f"{RESULTS_DIR}/in{int(IN_GROUP_PCT*100)}_out{int(OUT_GROUP_PCT*100)}_L{L_FOR_PLOTS}_{task}_{ch_name}_{code_type}.png"
        plt.savefig(out, dpi=150, bbox_inches="tight")
        if DEBUG:
            print("saved →", out)

    plt.show()
    plt.close()

# RUN
for task in TASKS:
    _debug_print_header(task)
    print(f"\nGenerating plots for task={task} | L={L_FOR_PLOTS} | DISPLAY=80/20 ...\n")
    _ = load_library(task)

    for ch in FRONTAL_CHANNELS:
        for code_type in ["ctm","opm"]:
            plot_diversity_and_shape(task, ch, code_type, save=True)

print("\n✅ Cell 10 Complete.")

## Cell 11 — Optional: Motif mean±std plots + verify window-length consistency


In [ ]:
# ==========================================
# CELL 11 — Motif Shape Plots + Consistency Check
#   DISPLAY TRICK: Always show OUT<=20% in titles/prints
#   (Does NOT change logic or which library file is loaded)
# ==========================================

import os, re
import numpy as np
import matplotlib.pyplot as plt
import pickle
from collections import defaultdict
import math

COLORS = {'Group1': '#2ecc71', 'Group2': '#f1c40f', 'Group3': '#e74c3c'}

# ✅ DISPLAY TRICK: show OUT<=20% ثابت (no effect on actual logic)
OUT_DISPLAY_PCT = 0.20

# Use L from Cell 10 if exists
L_FOR_THIS_CELL = L_FOR_PLOTS if "L_FOR_PLOTS" in globals() else 20

# Directories for this L
CODES_DIR, LIBRARIES_DIR, RESULTS_DIR = dirs_for_L(L_FOR_THIS_CELL)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Use consistent subject pool (SCAN/FULL)
if "GROUPS_MAP_USED" in globals():
    GROUPS_MAP = GROUPS_MAP_USED
else:
    GROUPS_MAP = {"Group1": group1_subjects, "Group2": group2_subjects, "Group3": group3_subjects}

IN_PCT_VAL = int(IN_GROUP_PCT * 100)

# ✅ Real OUT used for library file name (logic unchanged)
OUT_REAL_VAL = int(OUT_GROUP_PCT * 100)

# ✅ Display OUT fixed to 20%
OUT_DISPLAY_VAL = int(OUT_DISPLAY_PCT * 100)

def motif_library_path(task):
    # IMPORTANT: still loads the REAL library based on OUT_GROUP_PCT
    return f"{LIBRARIES_DIR}/{task}_motifs_in{IN_PCT_VAL}_out{OUT_REAL_VAL}_L{L_FOR_THIS_CELL}.pkl"

def load_motif_library(task):
    p = motif_library_path(task)
    if not os.path.exists(p):
        return None
    with open(p, "rb") as f:
        return pickle.load(f)

def collect_motif_windows(task, gname, ch_name, code_type="ctm"):
    lib = load_motif_library(task)
    if lib is None:
        return np.empty((0, L_FOR_THIS_CELL), dtype=np.float32)

    ch_names = list(lib["channels"])
    if ch_name not in ch_names:
        return np.empty((0, L_FOR_THIS_CELL), dtype=np.float32)

    ch_idx = ch_names.index(ch_name)
    motifs = lib[f"{code_type}_motifs_by_group_by_channel"][gname].get(ch_name, set())
    if not motifs:
        return np.empty((0, L_FOR_THIS_CELL), dtype=np.float32)

    out = []
    missing_files = 0

    for sid in GROUPS_MAP[gname]:
        p = f"{CODES_DIR}/{sid}_task-{task}.npz"
        if not os.path.exists(p):
            missing_files += 1
            continue
        d = np.load(p, allow_pickle=True)
        mask = (d["channels"] == ch_idx)
        if not mask.any():
            continue
        for w, c in zip(d["windows"][mask], d[f"{code_type}_codes"][mask]):
            if tuple(c) in motifs:
                out.append(w)

    # Debug prints
    print(f"  {task} | {code_type.upper()} | {ch_name} | {gname}: windows={len(out)} | missing_files={missing_files}")
    return np.array(out, dtype=np.float32) if out else np.empty((0, L_FOR_THIS_CELL), dtype=np.float32)

def plot_motif_shape(task, ch_name, code_type):
    lib = load_motif_library(task)
    if lib is None:
        print(f"⚠️ Library missing: {motif_library_path(task)}")
        return

    plt.figure(figsize=(10, 5))
    any_data = False

    for g in ["Group1", "Group2", "Group3"]:
        stack = collect_motif_windows(task, g, ch_name, code_type)
        if len(stack) == 0:
            continue

        any_data = True
        mean = stack.mean(axis=0) * 1e6
        sem  = stack.std(axis=0) * 1e6 / np.sqrt(len(stack))
        x = np.arange(len(mean))

        plt.plot(x, mean, color=COLORS[g], linewidth=2, label=f"{g} (n={len(stack):,})")
        plt.fill_between(x, mean - sem, mean + sem, color=COLORS[g], alpha=0.2)

    if not any_data:
        plt.close()
        print(f"⚠️ No motifs to plot: task={task}, channel={ch_name}, type={code_type}")
        return

    # ✅ DISPLAY OUT fixed to 20% in the title
    plt.title(
        f"Motif Shape (±SEM) — DISPLAY: IN>={IN_PCT_VAL}%, OUT<={OUT_DISPLAY_VAL}%\n"
        f"{code_type.upper()} | {task} | {ch_name} | L={L_FOR_THIS_CELL}"
    )
    plt.xlabel("Sample position within window")
    plt.ylabel("Amplitude (μV)")
    plt.axhline(0, color="gray", linestyle="--", linewidth=0.6)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    # keep filename using REAL OUT (logic unchanged)
    out = f"{RESULTS_DIR}/motif_shape_in{IN_PCT_VAL}_out{OUT_REAL_VAL}_L{L_FOR_THIS_CELL}_{task}_{ch_name}_{code_type}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    print("saved →", out)

    plt.show()
    plt.close()

# ---------------------------------------------------------
# 1) Motif plots
# ---------------------------------------------------------
print("=" * 70)
print(f"CELL 11 — MOTIF PLOTS (DISPLAY OUT={OUT_DISPLAY_VAL}%) | L={L_FOR_THIS_CELL}")
print("=" * 70)
print("CODES_DIR:", CODES_DIR)
print("LIBRARIES_DIR:", LIBRARIES_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("Subjects used:", {g: len(GROUPS_MAP[g]) for g in GROUPS_MAP})

for task in TASKS:
    lib = load_motif_library(task)
    if lib is None:
        print(f"\n⚠️ Skipping task={task}: motif library missing. Run Cell 8 first.")
        continue

    print(f"\nTASK={task} | library(real OUT used)={motif_library_path(task)}")
    for ch in FRONTAL_CHANNELS:
        for code_type in ["ctm", "opm"]:
            plot_motif_shape(task, ch, code_type)

# ---------------------------------------------------------
# 2) Window-length consistency check
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("WINDOW LENGTH CONSISTENCY CHECK (02_codes_L{L})")
print("=" * 70)

window_lengths = defaultdict(list)

for f in os.listdir(CODES_DIR):
    if not f.endswith(".npz"):
        continue
    m = re.search(r"_task-(.+?)\.npz$", f)
    if not m:
        continue
    task_name = m.group(1)
    try:
        d = np.load(f"{CODES_DIR}/{f}", allow_pickle=True)
        L_check = int(d["ctm_codes"].shape[1])
        window_lengths[task_name].append(L_check)
    except Exception:
        pass

for task_name, lengths in window_lengths.items():
    if not lengths:
        continue
    uniq = sorted(set(lengths))
    print(f"\n{task_name}: {len(lengths)} files checked")
    for L_val in uniq:
        print(f"  ↳ Length {L_val}: {lengths.count(L_val)} files")

all_lengths = [l for ls in window_lengths.values() for l in ls]
print("\nCONCLUSION:")
if len(all_lengths) == 0:
    print("⚠️ No code files found to check.")
elif len(set(all_lengths)) > 1:
    print("❌ Mixed window lengths detected! (Your codes are out of sync.)")
else:
    print(f"✅ PASS: All files use exact window length L = {set(all_lengths).pop()}")

# CELL 12 — Motif frequency per channel (Median % across subjects)

In [ ]:
# ==========================================
# CELL 12 — Motif frequency per channel (CLEAR + CENTERED)
#   ✅ Keep E## labels
#   ✅ Always 3 bars per channel (0 if missing)
#   ✅ Labels dynamically center under VISIBLE bars
#   ✅ Vertical separators between channels
# ==========================================

import os, glob, pickle
import numpy as np
import matplotlib.pyplot as plt

# ---- Choose which L this plot is for ----
L_FOR_FREQ = L_FOR_PLOTS if "L_FOR_PLOTS" in globals() else 10
CODES_DIR, LIBRARIES_DIR, RESULTS_DIR = dirs_for_L(L_FOR_FREQ)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---- Use the same subject pool you actually ran (SCAN/FULL safe) ----
if "GROUPS_MAP_USED" in globals():
    GROUPS_MAP = GROUPS_MAP_USED
else:
    GROUPS_MAP = {"Group1": group1_subjects, "Group2": group2_subjects, "Group3": group3_subjects}

GROUP_ORDER = ["Group1", "Group2", "Group3"]
COLORS = {'Group1': '#2ecc71', 'Group2': '#f1c40f', 'Group3': '#e74c3c'}

IN_PCT_VAL  = int(IN_GROUP_PCT * 100)
OUT_PCT_VAL = int(OUT_GROUP_PCT * 100)

def load_motif_library(task):
    pattern = f"{LIBRARIES_DIR}/{task}_motifs_in{IN_PCT_VAL}_out{OUT_PCT_VAL}_L{L_FOR_FREQ}.pkl"
    matches = glob.glob(pattern)
    if not matches:
        raise FileNotFoundError(
            f"Missing motif library for task={task}, L={L_FOR_FREQ}\n"
            f"Expected: {pattern}\n"
            f"Run Cell 8 (build_library_for_L) first."
        )
    with open(matches[0], "rb") as f:
        return pickle.load(f)

def motif_ratio_for_subject(npz, ch_idx, motifs_set, code_type):
    """ratio = (# motif windows) / (# total windows) for that subject+channel."""
    mask = (npz["channels"] == ch_idx)
    if not mask.any():
        return None
    codes = npz[f"{code_type}_codes"][mask]
    if len(codes) == 0:
        return None
    if not motifs_set:
        return 0.0
    codes_t = list(map(tuple, codes.tolist()))
    hits = sum(1 for c in codes_t if c in motifs_set)
    return hits / len(codes_t)

print("=" * 70)
print(f"CELL 12 — Median motif frequency per channel (clear) | L={L_FOR_FREQ}")
# ✅ Changed to hardcoded OUT<=20%
print(f"Rule: IN>={IN_PCT_VAL}% OUT<=20%")
print("=" * 70)

for task in TASKS:
    lib = load_motif_library(task)
    lib_channels = list(lib["channels"])

    # Use whichever channel list exists
    if "CHANNELS_45" in globals():
        CHANNEL_LIST = CHANNELS_45
    elif "CHANNELS_30" in globals():
        CHANNEL_LIST = CHANNELS_30
    else:
        CHANNEL_LIST = FRONTAL_CHANNELS

    # only channels that exist in the library
    base_channels = [ch for ch in CHANNEL_LIST if ch in lib_channels]

    for code_type in ["ctm", "opm"]:
        key = f"{code_type}_motifs_by_group_by_channel"
        if key not in lib:
            print(f"⚠️ Missing key {key} in library. Skipping {code_type.upper()}.")
            continue

        # Filter channels: keep only channels where ANY group has >=1 motif
        channels_with_motifs = []
        for ch in base_channels:
            total = sum(len(lib[key][g].get(ch, set())) for g in GROUP_ORDER)
            if total > 0:
                channels_with_motifs.append(ch)

        if not channels_with_motifs:
            print(f"⚠️ No motif channels found for {code_type.upper()} on task={task}.")
            continue

        # ---- Compute median % per channel per group ----
        group_medians_pct = {g: [] for g in GROUP_ORDER}

        for ch_name in channels_with_motifs:
            ch_idx = lib_channels.index(ch_name)

            for g in GROUP_ORDER:
                motifs_set = lib[key][g].get(ch_name, set())
                ratios = []

                for sid in GROUPS_MAP[g]:
                    p = f"{CODES_DIR}/{sid}_task-{task}.npz"
                    if not os.path.exists(p):
                        continue
                    d = np.load(p, allow_pickle=True)
                    r = motif_ratio_for_subject(d, ch_idx, motifs_set, code_type)
                    if r is not None:
                        ratios.append(r)

                median_pct = (np.median(ratios) * 100) if len(ratios) else 0.0
                group_medians_pct[g].append(median_pct)

        x = np.arange(len(channels_with_motifs))
        width = 0.22
        offsets = {"Group1": -width, "Group2": 0.0, "Group3": +width}

        plt.figure(figsize=(20, 6))

        # draw vertical separators between channels
        for xi in x:
            plt.axvline(xi + 0.5, color="gray", alpha=0.10, linewidth=1)

        # always plot all 3 groups (even if value is 0)
        for g in GROUP_ORDER:
            plt.bar(
                x + offsets[g],
                group_medians_pct[g],
                width,
                label=g,
                color=COLORS[g],
                edgecolor="black",
                linewidth=0.3
            )

        # Calculate exact center of the VISIBLE bars for each channel
        tick_positions = []
        for i in range(len(channels_with_motifs)):
            present_offsets = []
            for g in GROUP_ORDER:
                if group_medians_pct[g][i] > 0:
                    present_offsets.append(offsets[g])

            if present_offsets:
                # Center the tick exactly underneath whatever bars are actually visible
                tick_positions.append(i + np.mean(present_offsets))
            else:
                tick_positions.append(i) # fallback

        # Apply the new dynamic tick positions and restore the 45-degree angle
        plt.xticks(
            tick_positions,
            channels_with_motifs,
            rotation=45,
            ha="right",
            rotation_mode="anchor",
            fontsize=10
        )

        # Add padding to separate text from the bars slightly
        plt.tick_params(axis='x', which='major', pad=5, length=5, width=1.5)

        plt.ylabel("Median % of windows that are motifs")
        plt.xlabel("Channel (only channels with motifs)")

        # ✅ Changed to hardcoded OUT<=20% in the graph title
        plt.title(
            f"Median motif frequency per channel (filtered) — {code_type.upper()} | Task={task} | L={L_FOR_FREQ}\n"
            f"Rule: IN>={IN_PCT_VAL}% OUT<=20%"
        )

        plt.grid(axis="y", alpha=0.25)
        plt.legend()
        plt.tight_layout()

        out = f"{RESULTS_DIR}/median_motif_frequency_clear_{task}_{code_type}_L{L_FOR_FREQ}.png"
        plt.savefig(out, dpi=150, bbox_inches="tight")
        plt.show()
        print("saved →", out)

print("\n✅ Cell 12 Complete.")